# Consistent decoy — Phase 2
Select **GPU T4 ×2** and attach the extracted private dataset. Use **Save Version → Save & Run All**.
Outputs are written to `/kaggle/working/phase2`. Inspect `phase2-status.json` in the saved version: notebook execution success is not proof of completed training.
The 8-hour total budget leaves time for saving. Abrupt platform shutdown can still lose unsaved session output; download saved versions as backups.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
import torch

HOURS = 8.0  # Total budget including checks; leave time to save outputs.
RESUME_FROM = None  # Attach a previously saved phase2 directory to continue later.
INPUT = Path("/kaggle/input/datasets/wushi09/consistent-decoy/consistent-decoy")
WORK = Path("/kaggle/working")
started = time.monotonic()
WORK.mkdir(parents=True, exist_ok=True)
if not torch.cuda.is_available():
    raise RuntimeError("Select GPU T4 x2. This pipeline requires CUDA, not TPU.")

def stage_input(source, work):
    project = work / "consistent-decoy"
    if not (source / "EXPORT_MANIFEST.json").is_file():
        candidates = sorted(Path("/kaggle/input").rglob("EXPORT_MANIFEST.json"))
        if len(candidates) == 1:
            source = candidates[0].parent
        else:
            archives = sorted(Path("/kaggle/input").rglob("consistent-decoy-kaggle.zip"))
            if len(archives) != 1:
                raise RuntimeError(f"Cannot identify one bundle at {source}; found {len(candidates)} manifests and {len(archives)} archives")
            with zipfile.ZipFile(archives[0]) as z:
                for entry in z.infolist():
                    if not (work / entry.filename).resolve().is_relative_to(work.resolve()):
                        raise RuntimeError("Unsafe archive path")
                z.extractall(work)
            source = project
    manifest = json.loads((source / "EXPORT_MANIFEST.json").read_text())
    old = project / "EXPORT_MANIFEST.json"
    if old.exists() and json.loads(old.read_text())["snapshot"] != manifest["snapshot"]:
        raise RuntimeError("Existing working project is a different snapshot; preserve its outputs before changing projects")
    for rel, expected in manifest["sha256"].items():
        origin, target = source / rel, project / rel
        if not origin.resolve().is_relative_to(source.resolve()) or not target.resolve().is_relative_to(project.resolve()):
            raise RuntimeError("Unsafe manifest path")
        if hashlib.sha256(origin.read_bytes()).hexdigest() != expected:
            raise RuntimeError(f"Input snapshot hash mismatch: {rel}")
        if origin.resolve() != target.resolve():
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(origin, target)
    project.mkdir(parents=True, exist_ok=True)
    (project / "EXPORT_MANIFEST.json").write_text(json.dumps(manifest, indent=2))
    return project, manifest

project, manifest = stage_input(INPUT, WORK)
out=WORK/"phase2"
if RESUME_FROM:
    if out.exists(): raise RuntimeError("Output already exists; do not merge checkpoint trees")
    shutil.copytree(RESUME_FROM,out)
print("Snapshot",manifest["snapshot"],"torch",torch.__version__,"GPUs",torch.cuda.device_count())

def save_json(path, value):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(value, indent=2))
    os.replace(tmp, path)

jobs = []
state = {"snapshot": manifest["snapshot"], "status": "preflight", "output_directory": str(out)}
def inventory():
    files = [{"path": str(p.relative_to(WORK)), "bytes": p.stat().st_size}
             for p in out.rglob("*") if p.is_file() and p.suffix != ".tmp"] if out.exists() else []
    save_json(WORK/"output-inventory.json", {"files": files, "total_bytes": sum(f["bytes"] for f in files)})
    return files

def status():
    state["jobs"] = {plant: {"exit_code": p.poll(), "complete": (out/f"COMPLETE_{plant}.json").exists()}
                     for plant,p,_ in jobs}
    state["elapsed_hours"] = (time.monotonic()-started)/3600
    save_json(WORK/"phase2-status.json", state)
    inventory()

def check(args):
    with (WORK/"preflight.log").open("a") as log:
        result = subprocess.run([sys.executable,"-u",*args],cwd=project,stdout=log,stderr=subprocess.STDOUT)
    if result.returncode:
        print((WORK/"preflight.log").read_text()[-6000:])
        raise RuntimeError(f"Preflight failed: {args}; see preflight.log")
    print("Passed:", " ".join(args), flush=True)

try:
    status()
    check(["-m","unittest","discover","-s","tests"])
    check(["scripts/02_train.py","--smoke","--device","cuda","--out",str(WORK/"gpu-smoke")])
    check(["scripts/02_train.py","--gpu-preflight","--device","cuda"])
    if (time.monotonic()-started)/3600 >= HOURS:
        raise RuntimeError("Preflight exhausted the time budget; outputs saved, full training was not launched")
    plants=["hai","qappd"] if torch.cuda.device_count()>=2 else ["all"]
    state["status"]="training"
    for gpu,plant in enumerate(plants):
        env=os.environ.copy();env["CUDA_VISIBLE_DEVICES"]=str(gpu)
        log=(WORK/f"training-{plant}.log").open("a")
        remaining=max(.001,HOURS-(time.monotonic()-started)/3600)
        p=subprocess.Popen([sys.executable,"-u","scripts/02_train.py","--plant",plant,
            "--device","cuda","--hours",str(remaining),"--out",str(out)],
            cwd=project,env=env,stdout=log,stderr=subprocess.STDOUT)
        jobs.append((plant,p,log))
    while any(p.poll() is None for _,p,_ in jobs):
        status()
        for plant,p,_ in jobs:
            lines=(WORK/f"training-{plant}.log").read_text().splitlines()
            print(plant,"running" if p.poll() is None else f"exit={p.returncode}",lines[-1] if lines else "starting",flush=True)
        time.sleep(30)
    failed=[plant for plant,p,_ in jobs if p.returncode]
    complete=all((out/f"COMPLETE_{plant}.json").exists() for plant,p,_ in jobs)
    state["status"]="FAILED" if failed else "COMPLETE" if complete else "PAUSED_RESUMABLE"
    state["failed_jobs"]=failed
except Exception as error:
    # Preserve a successful notebook-version save path even when a training/preflight job fails.
    # The status file, rather than Kaggle's notebook exit alone, determines scientific completion.
    state["status"]="FAILED"
    state["error"]=repr(error)
    print("FAILED:",error,flush=True)
    for _,p,_ in jobs:
        if p.poll() is None:
            print("Waiting for the other job to finish its checkpointed time budget",flush=True)
            p.wait()
finally:
    for _,_,log in jobs:log.close()
    status()
    files=inventory()
    print("FINAL STATUS:",state["status"],flush=True)
    print("Saved output files:",len(files),"under",out,flush=True)
    print("After Save & Run All finishes, open the saved version's Output tab and download phase2 plus logs/status.",flush=True)
    print("COMPLETE means all requested training finished. PAUSED_RESUMABLE requires another session. FAILED requires inspecting logs.",flush=True)
